# 00 · Empieza aquí: de vídeo de retransmisión a datos tipo SkillCorner

Este cuaderno procesa un clip de prueba de 6 segundos (Chelsea–Barcelona 2009) y produce
exactamente lo que entrega SkillCorner: la posición en metros de cada jugador y del balón
10 veces por segundo, con identidades, equipos y el área que ve la cámara.

**En Google Colab:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*,
y luego *Entorno de ejecución → Ejecutar todas*.
**En tu Mac (Jupyter / Anaconda):** instala primero siguiendo el README y ejecuta las celdas en orden
con *Shift + Enter*.

In [ ]:
# 1) INSTALACIÓN — en Colab tarda ~2 min; en tu Mac (ya instalado) no hace nada.
import sys, subprocess, os
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not os.path.exists("tracking"):
        # rama con el código; si ya está fusionada en main, git usa la rama por defecto
        r = subprocess.run(["git", "clone", "-q", "-b", "claude/soccernet-calibration-pkg-r8517j", "https://github.com/delioguzmang-maker/tracking"])
        if r.returncode != 0:
            subprocess.run(["git", "clone", "-q", "https://github.com/delioguzmang-maker/tracking"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "tracking[notebooks]"], check=True)
    sys.path.insert(0, os.path.abspath("tracking"))
import soccercal
print("soccercal", soccercal.__version__, "listo")

## 2) Comprobar que todo funciona
`doctor` revisa Python, PyTorch, el acelerador (GPU en Colab, **MPS** en un MacBook con chip M),
descarga los pesos de las redes (una sola vez, ~300 MB) y hace una inferencia de prueba.
Si algo sale `[FALLO]`, el mensaje dice qué falta.

In [ ]:
from soccercal.cli import main as soccercal_cli
soccercal_cli(["doctor"])

## 3) Procesar el clip de prueba
La primera pasada ejecuta las redes neuronales (lo lento: ~15 s en Colab GPU, ~40 s en un Mac M1/M2,
varios minutos en CPU). Queda guardada en `salida_demo/analysis.pkl.gz`: si vuelves a ejecutar la
celda, se reutiliza y todo lo demás tarda segundos.

In [ ]:
import soccercal
from soccercal import Config

video = soccercal.get_sample_video()          # descarga el clip de 6 s
cfg = Config(home_name="Chelsea", away_name="Barcelona")
res = soccercal.run(video, "salida_demo", cfg)   # analiza + calibra + sigue + exporta + vídeo
res.summary()

## 4) Verificar con tus ojos (lo más importante)
Arriba: la retransmisión con **el campo dibujado en magenta a partir de la calibración** —si las
líneas magenta caen sobre las líneas reales, la calibración es buena— y cada persona con su
identidad. Abajo: el minimapa estilo SkillCorner (relleno = visto por la cámara, hueco = extrapolado;
zona gris = lo que ve la cámara).

In [ ]:
import cv2, matplotlib.pyplot as plt
cap = cv2.VideoCapture("salida_demo/verificacion.mp4")
fig, axes = plt.subplots(1, 2, figsize=(16, 9))
for ax, n in zip(axes, (10, 140)):
    cap.set(cv2.CAP_PROP_POS_FRAMES, n); ok, fr = cap.read()
    ax.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); ax.axis("off"); ax.set_title(f"fotograma {n}")
plt.tight_layout(); plt.show()

In [ ]:
# Ver el vídeo completo dentro del cuaderno (en Colab o Jupyter)
from base64 import b64encode
from IPython.display import HTML
import subprocess, shutil
src = "salida_demo/verificacion.mp4"
# mp4v no se reproduce en todos los navegadores: se convierte a H.264 si hay ffmpeg
if shutil.which("ffmpeg"):
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-vcodec", "libx264", "-pix_fmt", "yuv420p",
                    "salida_demo/verificacion_h264.mp4"])
    src = "salida_demo/verificacion_h264.mp4"
HTML(f'<video width=800 controls src="data:video/mp4;base64,{b64encode(open(src, "rb").read()).decode()}"></video>')

## 5) Los datos
`res.table` tiene una fila por jugador y fotograma (10 fps): posición `x, y` en metros
(origen en el centro, `x` a lo largo del campo, `y` hacia la banda lejana, igual que SkillCorner),
`is_detected` (visto por la cámara o extrapolado) y velocidad.

In [ ]:
res.table.head(10)

In [ ]:
from soccercal.viz import plot_frame
plot_frame(res, k=30);

## 6) Métricas físicas (distancias por bandas de velocidad, velocidad máxima, sprints)

In [ ]:
res.physical.round(1)

## 7) Formato SkillCorner
En `salida_demo/` están `1_match.json` y `1_tracking_extrapolated.jsonl`, el mismo formato que los
datos abiertos de SkillCorner. Se pueden abrir con su visor (`SkillCorner_Tracking_Viewer.html` del
repositorio `SkillCorner/opendata`) o con **kloppy**, la librería estándar de datos de fútbol:

In [ ]:
from soccercal.skillcorner import load_kloppy
ds = load_kloppy("salida_demo")
print(ds.metadata.provider, ds.metadata.frame_rate, "fps,", len(ds.records), "fotogramas")
ds.to_df().head()

In [ ]:
import os
print("\n".join(sorted(os.listdir("salida_demo"))))
if EN_COLAB:   # descargar todo en un zip
    import shutil
    from google.colab import files
    shutil.make_archive("salida_demo", "zip", "salida_demo")
    files.download("salida_demo.zip")